In [77]:
# Import custom mahjong classes
from helper import config
from helper.tile_util import Tile, MahjongMeld
from helper.utility import Hand136, MSPZD, MahjongConverter
from helper.game_util import GameLogEntry, GamePhase
from helper.player import MahjongPlayer
from helper.game import MahjongGame
from helper.visualizer import MahjongReplay, VizMode

from typing import List, Dict, Set, Tuple, Union

from mahjong.hand_calculating.hand import HandCalculator
from mahjong.hand_calculating.hand_config import HandConfig
from mahjong.shanten import Shanten
from mahjong.tile import TilesConverter
from mahjong.agari import Agari

from dataclasses import dataclass
import heapq

from time import time
import random
import matplotlib.pyplot as plt
import numpy as np
import copy
import math

In [78]:
def setup_player(name: str) -> Tuple[MahjongPlayer, List[int], List[int], int, List[int]]:
    """
    Setup a single instace of MahjongPlayer object to play a single turn in a single kyoku.
    Returns player, wall, dead_wall, draw_tile, dora_indicators
    """
    player = MahjongPlayer(0, name)
    player.reset_for_kyoku(seat_wind=0, is_dealer=True)
    all_tiles = list(range(136))
    # Allow red dora
    random.shuffle(all_tiles)

    # Reserve dead wall (last 14 tiles)
    dead_wall = all_tiles[-MahjongGame.DEAD_WALL_SIZE:]
    wall = all_tiles[:-MahjongGame.DEAD_WALL_SIZE]

    # give out hand
    player.set_hand([wall.pop() for _ in range(13)])

    draw_tile = wall.pop()

    dora_indicators = [wall.pop()]

    return player, wall, dead_wall, draw_tile, dora_indicators

# Simulating the simulation
- Simulate individual sections / algorithms for solving the rii-chi mahjong game theory
- Below code cell is for checking functionalities
  - Main functionality in mahjong library are...
    - `HandCalculator.estimate_hand_value(**, HandConfig())`
    - `Shanten.calculate_shanten(**, use_chiitoitsu, use_kokushi)`

In [79]:
player, wall, dead_wall, draw_tile, dora_indicators = setup_player("Greedy")
hand_136 = player.hand
draw_136 = Hand136(draw_tile)
hand_34 = hand_136.to_34()
draw_34 = draw_136.to_34()
hand_mspzd = hand_136.to_mspzd(True)
draw_mspzd = draw_136.to_mspzd(True)
# combined len==14
comb_hand_136 = hand_136+draw_136
comb_hand_34 = comb_hand_136.to_34()
comb_hand_mspzd = comb_hand_136.to_mspzd()

print("Current Hand: ", hand_136.ids, "->", hand_mspzd)
print(f"Drawn tile: {draw_136.ids} -> {draw_mspzd}")

hand_calculator = HandCalculator()
config = HandConfig()

result = hand_calculator.estimate_hand_value(
    tiles = comb_hand_136,
    win_tile = draw_tile,
    melds = player._to_library_meld_tuples(),
    dora_indicators = dora_indicators,
    config = config
)
print(f"HandCalculator.estimate_hand_value() = {result}")

# Get Shanten of current hand
shanten = Shanten()
num_shanten = shanten.calculate_shanten( # Calculate minimum shanten[int]
    tiles_34 = comb_hand_34,
    use_chiitoitsu = False,
    use_kokushi = False
)
print(f"Current shanten: Shanten.calculate_shanten() = {num_shanten}")

Current Hand:  [7, 39, 50, 54, 63, 67, 72, 73, 89, 92, 96, 109, 135] -> 2m14578p11567sEChunz
Drawn tile: [51] -> 4p
HandCalculator.estimate_hand_value() = hand_not_winning
Current shanten: Shanten.calculate_shanten() = 3


# Greedy Algo
1. Given depth $d$, a hand in 136 format of `len=14`
2. Replace each tile in hand with existing tile from wall
3. Recurse
4. $E = p*\mathbb{shanten}$ Where $E$ is Expected value, $p$ is the probability of pulling that tile

### Search Space Math
**Search Space**

Given $T = 34$, 4 tiles per $T$, depth $d$

Then search space (leaf count) is roughly $N(d) = {476}^d$

**Uke ires**

Given $S(H)$ is $shanten$ for a hand $H$, then for each tile type $t \in T$

Ukeire is if $S(H) < S(H+t)$




### Time
Around 2.5s / 100 Branches. *NOT REALISTIC*

In [80]:
# Greedy N-depth search on which hand is best for Shanten

class SearchStep:
    def __init__(self, depth, discard=None, draw=None, prob=1.0, shanten=None, ukeire=None):
        self.depth = depth
        self.discard = discard
        self.draw = draw
        self.prob = prob
        self.shanten = shanten
        self.ukeire = ukeire

    def _to_mspzd(self, n):
        return Hand136(n).to_mspzd()

    def __repr__(self):
        return f"\n[D:{self.depth}, discard:{self._to_mspzd(self.discard)}, draw:{self._to_mspzd(self.draw)}, shanten:{self.shanten}, ukeire:{self.ukeire}, P:{self.prob:.4f}]"


class SearchResult:
    def __init__(self, score, path):
        self.score = score        # expected shanten score
        self.path = path          # list[SearchStep]

    def __repr__(self):
        return f"Score:{self.score}, path:{self.path}"


class GreedyShantenSearch:

    UKEIRE_COEFF = 0.01

    def __init__(self):
        self.shanten = Shanten()
        self.NUM_BRANCHES = 0

    def _get_shanten(self, hand_136: Hand136) -> int:
        """Hand must contain 14 tiles"""
        return self.shanten.calculate_shanten(hand_136.to_34())

    def _analyze_draws(self, hand_136: Hand136, wall_counts: List[int]):
        """
        For a 13-tile hand simulate all possible draws.
        Returns list of (tile_type, remaining, new_shanten)
        """
        results = []

        for tile_type in range(34):

            remaining = wall_counts[tile_type]
            if remaining == 0:
                continue

            draw_tile = tile_type * 4

            new_hand = copy.copy(hand_136)
            new_hand = new_hand.add(draw_tile)

            new_shanten = self._get_shanten(new_hand)

            results.append((tile_type, remaining, new_shanten))

        return results

    def _get_ukeire(self, hand_136: Hand136, wall_counts: List[int]) -> int:
        """
        hand_136 must contain 13 tiles
        """
        current_shanten = self.shanten.calculate_shanten(hand_136.to_34())

        ukeire = 0
        draws = self._analyze_draws(hand_136, wall_counts)

        for _, remaining, new_shanten in draws:
            if new_shanten < current_shanten:
                ukeire += remaining

        return ukeire

    def _evaluate(self, hand_136: Hand136, wall_counts: List[int]) -> float:
        """
        Evaluate a 14-tile hand
        """
        self.NUM_BRANCHES += 1
        if self.NUM_BRANCHES % 100 == 0:
            print(f"Current Branch: {self.NUM_BRANCHES}")

        shanten = self._get_shanten(hand_136)

        # convert to 13 tile state for ukeire calculation
        best_ukeire = 0

        for discard in list(hand_136):
            hand13 = copy.copy(hand_136)
            hand13 = hand13.remove(discard)

            ukeire = self._get_ukeire(hand13, wall_counts)
            best_ukeire = max(best_ukeire, ukeire)

        return shanten - GreedyShantenSearch.UKEIRE_COEFF * best_ukeire

    def search(
        self,
        hand_136: Hand136,
        wall_counts: List[int],
        depth: int,
        prob: float = 1.0,
        path=None
    ) -> Union[SearchResult, None]:

        if path is None:
            path = []

        # BASE CASE
        if depth == 0:
            return SearchResult(
                score=self._evaluate(hand_136, wall_counts),
                path=path
            )

        best_result = None
        best_score = -math.inf

        current_shanten = self._get_shanten(hand_136)

        for discard in list(hand_136):

            # ---- DISCARD ----
            hand_after_discard = copy.copy(hand_136)
            hand_after_discard = hand_after_discard.remove(discard)  # 13 tiles

            ukeire = self._get_ukeire(hand_after_discard, wall_counts)

            discard_step = SearchStep(
                depth=depth,
                discard=discard,
                prob=prob,
                ukeire=ukeire
            )

            total_tiles = sum(wall_counts)

            expected_score = 0
            best_child_result = None

            # ---- DRAW ANALYSIS ----
            draws = self._analyze_draws(hand_after_discard, wall_counts)

            for tile_type, remaining, new_shanten in draws:
                # Prune branches which worsens shanten
                if new_shanten < current_shanten:
                    continue

                draw_prob = remaining / total_tiles
                draw_tile = tile_type * 4

                new_hand = copy.copy(hand_after_discard)
                new_hand = new_hand.add(draw_tile)

                new_wall = wall_counts.copy()
                new_wall[tile_type] -= 1

                draw_step = SearchStep(
                    depth=depth,
                    draw=draw_tile,
                    prob=prob * draw_prob,
                    shanten=new_shanten
                )

                result = self.search(
                    new_hand,
                    new_wall,
                    depth - 1,
                    prob * draw_prob,
                    path + [discard_step, draw_step]
                )

                expected_score += draw_prob * -result.score

                if best_child_result is None or result.score > best_child_result.score:
                    best_child_result = result

            if expected_score > best_score:
                best_score = expected_score
                best_result = SearchResult(
                    score=expected_score,
                    path=best_child_result.path
                )

        return best_result
    
player, wall, dead_wall, draw_tile, dora_indicators = setup_player("Greedy")
shanten_calculator = Shanten()
greedy_searcher = GreedyShantenSearch()
comb_hand = player.hand.add(draw_tile)
depth = 1
# best_result = greedy_searcher.search(comb_hand, wall.copy(), depth)

# print(f"Current hand:{player.hand.to_mspzd()}, Drawn tile:{Hand136(draw_tile).to_mspzd()}")
# print(f"Combined len==14 Hand:{comb_hand.to_mspzd()}, Shanten:{shanten_calculator.calculate_shanten(comb_hand.to_34())}")
# print(f"Greedy search results:{best_result.path}")
# print(f"Results:{best_result.score}, Depth:{depth}, Num Search Branch:{greedy_searcher.NUM_BRANCHES}")

## Beam Search
- Given $B$ as beam width, $D$ as search depth, $A$ as avg number of actions (Branching factor)
- then $O(B\times A\times D)$ for speed, $O(B)$ for memory
- For Mahjong w/o pruning, $A = 34\times 14 = 476$

## Shape Efficiency
- Shanten improvement > Ukeire is not good enough. Consider _Shapes_ which are good in this order.
  - to avoid double counting, count only the longest / strongest shape
Solution, Heuristics (e.g. scoring for shapes) in order of good
- 3men e.g. 34567p waiting for 258p
- double ryanmen e.g. 3344s waiting for 25s x2
- pair in a meld e.g. 2334
- 4tile straight e.g. 2345
- pair of honor tiles with yaku e.g. EE, SS, ChunChun where E is prevalent wind and S is self wind
- ryanmen e.g. 45s waiting for 36s
- pair e.g. 22s
- complete melds e.g. 333s
- complete incrementive melds e.g. 234s
- pair and ryamen e.g. 334, NN (not yaku wind)
- middle kanchan e.g. 68m waiting for 7m
- penchan e.g. 89p waiting for 7p
- pen kanchan e.g. 13m waiting for 2m
**BAD SHAPES**
- isolated 1,9
- isolated yakuhai
  - dragons, prevalent wind, self wind is better
- isolated middle tiles 3-7
- tiles not connected to neighbours e.g. 12589m -> 5m is floating

In [88]:
from dataclasses import dataclass
from typing import List, Tuple, Dict
import heapq

@dataclass
class BeamNode:
    hand34: Tuple[int, ...]        # 34-count tuple for 13-tile hand
    wall_key: Tuple[int, ...]      # 34-count tuple for remaining wall
    score: float
    path: list

class MahjongBeamSearch:
    VALUE_COEFF = 1.0
    UKEIRE_COEFF = 0.05
    SHAPE_COEFF = 0.02
    DISCARD_YAKUHAI_PENALTY = 30.0

    def __init__(self, beam_width:int, depth:int, self_wind:int, prevalent_wind:int):
        self.beam_width = beam_width
        self.max_depth = depth
        self.self_wind = self_wind
        self.prevalent_wind = prevalent_wind

        self.shanten = Shanten()

        # caches
        self._shanten_cache: Dict[Tuple[int,...], int] = {}
        self._shape_cache: Dict[Tuple[int,...], float] = {}
        self._value_cache: Dict[Tuple[int,...], float] = {}
        self._eval_cache: Dict[Tuple[Tuple[int,...], Tuple[int,...]], float] = {}

        self.nodes_expanded = 0
        self._counter = 0

    # -----------------------------
    # small helpers (34-array ops)
    # -----------------------------
    @staticmethod
    def hand34_add_tile(hand34: Tuple[int,...], tile34: int) -> Tuple[int,...]:
        lst = list(hand34)
        lst[tile34] += 1
        return tuple(lst)

    @staticmethod
    def hand34_remove_tile(hand34: Tuple[int,...], tile34: int) -> Tuple[int,...]:
        lst = list(hand34)
        lst[tile34] -= 1
        return tuple(lst)

    @staticmethod
    def is_honor(tile34:int) -> bool:
        return 27 <= tile34 <= 33

    def is_yakuhai_tile34(self, tile34:int) -> bool:
        if tile34 in (31,32,33):  # dragons
            return True
        if tile34 == self.self_wind or tile34 == self.prevalent_wind:
            return True
        return False

    # -----------------------------
    # Cached shanten (34-array)
    # -----------------------------
    def get_shanten_from34(self, hand34: Tuple[int,...]) -> int:
        cached = self._shanten_cache.get(hand34)
        if cached is not None:
            return cached
        s = self.shanten.calculate_shanten(list(hand34))
        self._shanten_cache[hand34] = s
        return s

    # -----------------------------
    # Shape scoring (works on 34-array)
    # -----------------------------
    def shape_score_from34(self, hand34: Tuple[int,...]) -> float:
        cached = self._shape_cache.get(hand34)
        if cached is not None:
            return cached

        score = 0
        hand = list(hand34)

        for suit_start in (0,9,18):
            temp = hand[suit_start:suit_start+9].copy()

            # triplets
            for i in range(9):
                while temp[i] >= 3:
                    score += 5
                    temp[i] -= 3

            # sequences
            for i in range(7):
                while temp[i] >= 1 and temp[i+1] >= 1 and temp[i+2] >= 1:
                    score += 5
                    temp[i] -= 1
                    temp[i+1] -= 1
                    temp[i+2] -= 1

            # ryanmen (1..6)
            for i in range(1,7):
                while temp[i] >= 1 and temp[i+1] >= 1:
                    score += 7
                    temp[i] -= 1
                    temp[i+1] -= 1

            # kanchan
            for i in range(7):
                while temp[i] >= 1 and temp[i+2] >= 1:
                    score += 4
                    temp[i] -= 1
                    temp[i+2] -= 1

            # penchan edges
            if temp[0] >= 1 and temp[1] >= 1:
                score += 2
                temp[0] -= 1
                temp[1] -= 1
            if temp[7] >= 1 and temp[8] >= 1:
                score += 2
                temp[7] -= 1
                temp[8] -= 1

            # pairs
            for i in range(9):
                if temp[i] >= 2:
                    score += 6
                    temp[i] -= 2

            # isolated singles
            for i in range(9):
                if temp[i] == 1:
                    left2  = temp[i-2] if i >= 2 else 0
                    left1  = temp[i-1] if i >= 1 else 0
                    right1 = temp[i+1] if i <= 7 else 0
                    right2 = temp[i+2] if i <= 6 else 0
                    if left2 == left1 == right1 == right2 == 0:
                        score -= 5
                    temp[i] -= 1

        # honors
        for i in range(27,34):
            if hand[i] >= 3:
                score += 10
            elif hand[i] >= 2:
                score += 8

        self._shape_cache[hand34] = score
        return score

    # -----------------------------
    # Value / yakuhai scoring (34-array)
    # -----------------------------
    def value_score_from34(self, hand34: Tuple[int,...]) -> float:
        cached = self._value_cache.get(hand34)
        if cached is not None:
            return cached
        v = 0
        counts = hand34
        # dragons
        for d in (31,32,33):
            if counts[d] >= 3:
                v += 40
            elif counts[d] >= 2:
                v += 25
        # seat/prevalent winds
        for w in (self.self_wind, self.prevalent_wind):
            if w is None:
                continue
            if counts[w] >= 3:
                v += 36
            elif counts[w] >= 2:
                v += 20
        self._value_cache[hand34] = v
        return v

    # -----------------------------
    # Fast ukeire (honor-aware, normalized)
    # -----------------------------
    def get_fast_ukeire_from34(self, hand34: Tuple[int,...], wall_counts: Tuple[int,...]) -> float:
        base_shape = self.shape_score_from34(hand34)
        improvement = 0.0
        total = max(1, sum(wall_counts))
        counts = hand34
        for t in range(34):
            rem = wall_counts[t]
            if rem == 0:
                continue
            # shape improvement
            new_hand34 = list(counts)
            new_hand34[t] += 1
            new_hand34 = tuple(new_hand34)
            new_shape = self.shape_score_from34(new_hand34)
            if new_shape > base_shape:
                improvement += rem
            # honor completion
            if self.is_honor(t) and counts[t] == 2:
                improvement += 4 * rem
            if self.is_honor(t) and counts[t] == 1:
                improvement += 1 * rem
        return improvement / total

    # -----------------------------
    # Evaluate a 14-tile hand represented as 34-array (post-draw)
    # returns best immediate-discard score (memoized)
    # -----------------------------
    def evaluate_node_from34(self, hand14_34: Tuple[int,...], wall_counts: Tuple[int,...]) -> float:
        key = (hand14_34, wall_counts)
        cached = self._eval_cache.get(key)
        if cached is not None:
            return cached

        best = -1e9
        # iterate possible discards by tile34 where count>0
        for t in range(34):
            if hand14_34[t] == 0:
                continue
            # remove one tile t -> 13-tile hand
            hand13 = list(hand14_34)
            hand13[t] -= 1
            hand13 = tuple(hand13)

            shanten = self.get_shanten_from34(hand13)
            shape = self.shape_score_from34(hand13)
            ukeire = self.get_fast_ukeire_from34(hand13, wall_counts)
            value = self.value_score_from34(hand13)

            penalty = self.DISCARD_YAKUHAI_PENALTY if self.is_yakuhai_tile34(t) else 0.0

            score = (
                - shanten
                + self.VALUE_COEFF * value
                + self.SHAPE_COEFF * shape
                + self.UKEIRE_COEFF * ukeire
                - penalty
            )
            if score > best:
                best = score

        self._eval_cache[key] = best
        return best

    # -----------------------------
    # Beam Search (optimized, memoized expected value)
    # -----------------------------
    def search(self, start_hand: "Hand136", wall_counts_list: List[int]):
        # convert inputs to 34-tuple keys
        start_hand34 = tuple(start_hand.to_34())
        wall_key = tuple(wall_counts_list)
        total_tiles = sum(wall_key)

        start_score = self.evaluate_node_from34(start_hand34, wall_key)
        beam = [ BeamNode(hand34=start_hand34, wall_key=wall_key, score=start_score, path=[]) ]

        for _depth in range(self.max_depth):
            heap = []
            for node in beam:
                hand13_34 = node.hand34
                wall = list(node.wall_key)
                # iterate discards (tile types present)
                for discard_t in range(34):
                    if hand13_34[discard_t] == 0:
                        continue
                    # 13-tile after discard
                    hand_after_discard = list(hand13_34)
                    hand_after_discard[discard_t] -= 1
                    hand_after_discard = tuple(hand_after_discard)

                    # expected value over draws (memoized evaluate_node_from34)
                    expected = 0.0
                    best_draw_tile = None
                    best_post_value = -1e9
                    if total_tiles == 0:
                        continue
                    for tile_type, remaining in enumerate(wall):
                        if remaining == 0:
                            continue
                        # new 14-tile hand34
                        new_hand14 = list(hand_after_discard)
                        new_hand14[tile_type] += 1
                        new_hand14 = tuple(new_hand14)

                        # new wall after draw
                        new_wall = wall.copy()
                        new_wall[tile_type] -= 1
                        new_wall_key = tuple(new_wall)

                        draw_prob = remaining / total_tiles
                        post_value = self.evaluate_node_from34(new_hand14, new_wall_key)
                        expected += draw_prob * post_value

                        # track best post-draw tile (representative)
                        if post_value > best_post_value:
                            best_post_value = post_value
                            best_draw_tile = tile_type

                    # convert tile34 -> tile136 representative (use first copy)
                    rep_draw_tile136 = (best_draw_tile * 4) if best_draw_tile is not None else None
                    rep_discard_tile136 = discard_t * 4

                    new_node = BeamNode(
                        hand34=hand_after_discard,
                        wall_key=tuple(wall),
                        score=expected,
                        path=node.path + [(rep_discard_tile136, rep_draw_tile136, expected)]
                    )
                    self._counter += 1
                    heapq.heappush(heap, (-expected, self._counter, new_node))
                    if len(heap) > self.beam_width:
                        heapq.heappop(heap)
                    self.nodes_expanded += 1

            beam = [n for _,_,n in heap]

        return beam

    # -----------------------------
    # Logging
    # -----------------------------

    def print_results(self, beam, start_hand: Hand136):
        print("\n====== BEAM SEARCH RESULT ======")
        print("Nodes Expanded:", self.nodes_expanded)
        print("Beam Width:", self.beam_width)

        best = max(beam, key=lambda x: x.score)
        print("\nBest Score:", best.score)

        # start hand as 34-count tuple
        hand34 = tuple(start_hand.to_34())
        print("\nInitial Hand:", start_hand.to_mspzd())
        print("\n")
        print(f"{'Step':<5} | {'Discard':<8} | {'Draw':<8} | {'Score':<10} | {'Shanten':<8} | {'Hand'}")
        print("-"*75)

        # best.path expected entries: (discard_tile136, draw_tile136, expected)
        for i, entry in enumerate(best.path):
            # support older formats: (discard34, expected) or (discard136, draw136, expected)
            if len(entry) == 3:
                discard136, draw136, score = entry
                discard34 = discard136 // 4 if discard136 is not None else None
                draw34 = draw136 // 4 if draw136 is not None else None
            elif len(entry) == 2:
                discard34, score = entry
                draw34 = None
            else:
                # fallback: assume single tile34 stored
                discard34 = entry[0]
                draw34 = None
                score = entry[1] if len(entry) > 1 else 0.0

            # apply discard/draw on 34-array safely
            if discard34 is not None:
                if hand34[discard34] <= 0:
                    # defensive: if count is zero, skip decrement to avoid negative counts
                    pass
                else:
                    lst = list(hand34)
                    lst[discard34] -= 1
                    hand34 = tuple(lst)

            if draw34 is not None:
                lst = list(hand34)
                lst[draw34] += 1
                hand34 = tuple(lst)

            
            hand_str = Hand136(MahjongConverter().from_34_to_136(hand34)).to_mspzd()

            # compute shanten on the 34-array (safe)
            shanten = self.get_shanten_from34(hand34)

            discard_str = Hand136(discard34*4).to_mspzd().notation if discard34 is not None else "—"
            draw_str = Hand136(draw34*4).to_mspzd().notation if draw34 is not None else "—"

            print(f"{i:<5} | {discard_str:<8} | {draw_str:<8} | {score:<10.4f} | {shanten:<8} | {hand_str}")




player, wall, dead_wall, draw_tile, dora_indicators = setup_player("Greedy")
wall_counts = [0] * 34
for tid in wall:
    wall_counts[tid // 4] += 1
from helper import config
beam_engine = MahjongBeamSearch(beam_width=10, depth=5, self_wind = config.SOUTH, prevalent_wind = config.EAST)
comb_hand = player.hand.add(draw_tile)
st = time()

beam = beam_engine.search(comb_hand, wall_counts)
print(f"Time taken:{time()-st}")

print(f"hand:{comb_hand.to_mspzd()}")

beam_engine.print_results(beam, comb_hand)

"""
w=, d=, time=
"""

Time taken:10.73968768119812
hand:3679m256667p1236s

====== BEAM SEARCH RESULT ======
Nodes Expanded: 395
Beam Width: 10

Best Score: -4.612037559513312

Initial Hand: 3679m256667p1236s


Step  | Discard  | Draw     | Score      | Shanten  | Hand
---------------------------------------------------------------------------
0     | 7m       | 3p       | -2.4657    | 2        | 369m2356667p1236s
1     | 7p       | 6p       | -3.5691    | 2        | 369m2356666p1236s
2     | 3s       | 6p       | -2.6026    | 3        | 369m2356666p126s
3     | 1s       | 6p       | -3.6798    | 5        | 369m2356666p26s
4     | 6p       | 5p       | -4.6120    | 4        | 369m23556666p26s


'\nw=, d=, time=\n'

## Accounting for others
Currently, the above beam search algorithm only looks for optimal discard from hand. We must still take into consideration...
- Pon, Kan, Chi > Ensure it already has a yaku, or a valid yaku is likely
  - single suit, self wind, prevalent wind, haku, hatsu, chun, toitoi, kuitan, etc...
- Assume opponent's Tenpai
- Given opponent is Tenpai, predict their worse case score from discarded tiles and their furiten, and rank currend hand's tile's safety level

In [ ]:
player, wall, dead_wall, draw_tile, dora_indicators = setup_player("Greedy")
